# Romanian-Transformers Model Evaluation

Use this colab to evaluate _tranformer model performance_. We currently support:

*   [**Named Entity Recognition**](https://github.com/dumitrescustefan/ronec), based on RONECv2
*   [**Part of Speech Tagging**](https://github.com/dumitrescustefan/ro-pos-tagger), based on UD's Ro-RRT dataset
*   [**Semantic Textual Simiarity**](https://github.com/dumitrescustefan/RO-STS), based on RO-STS
*   [**Emotion Detection in Tweets**](https://github.com/Alegzandra/RED-Romanian-Emotions-Dataset), based on REDv2
*   [**Perplexity**](https://github.com/dumitrescustefan/wiki-ro), based on wiki-ro, only for generative models

**How to use:**
1. Choose the model in the dropdown below.
2. Choose the task
3. Choose the number of iterations (how many times to run the same task and average over)
4. Run all cells ($\color{red}{\text{Ctrl+F9}}$); you will see the averaged results printed at the bottom, as well as saved as jsons in each task's respective folder.

** Note: Use a GPU runtime and a browser addon (like Colab Auto Reconnect) to keep this session open - some tasks, with 5 iterations, might take some hours to complete.
This is the official script used to measure model performance on
the [Romanian-Transformers repo](https://github.com/dumitrescustefan/Romanian-Transformers).


---



In [ ]:
#@title Evaluation parameters

model = 'bert-base-multilingual-cased' #@param ["dumitrescustefan/gpt-neo-romanian-780m", "dumitrescustefan/bert-base-romanian-cased-v1","dumitrescustefan/bert-base-romanian-uncased-v1", "racai/distilbert-base-romanian-cased", "readerbench/RoGPT2-base", "readerbench/RoGPT2-medium", "readerbench/RoGPT2-large", "xlm-roberta-base", "bert-base-multilingual-cased", "bert-base-multilingual-uncased", "readerbench/RoBERT-small", "readerbench/RoBERT-base", "readerbench/RoBERT-large"] {allow-input: true}
task = 'Named Entity Recognition' #@param ["Named Entity Recognition", "POS Tagging", "Emotion Detection in Tweets", "Semantic Textual Similarity", "CLM Perplexity"]
iterations = '1' #@param ["1", "2", "3", "5"]

print("\nWe are going to eval \033[92m{}\033[0m on the \033[92m{}\033[0m task for \033[92m{}\033[0m iteration(s).\n".format(model, task, iterations))

import torch
if not torch.cuda.is_available():
  print(f"\033[101m*** Please use a GPU-enabled colab! ***\033[0m")
else:
  print(f"\nRunning on a \033[92m{torch.cuda.get_device_name(0)}\033[0m with \033[92m{torch.cuda.get_device_properties(0).total_memory/1024/1024/1024:.0f}GB\033[0m RAM.")


We are going to eval bert-base-multilingual-cased on the Named Entity Recognition task for 1 iteration(s).


Running on a Tesla T4 with 15GB RAM.


##### Code section, run every cell automatically with Ctrl+F9

In [ ]:
# "robert" | "mbert"
MODEL_KEY  = "mbert"
# "diac" | "nodiac"
TRAIN_COND = "nodiac"

RUN_NAME = f"{MODEL_KEY}_train_{TRAIN_COND}"

MODEL_HUB = {
    "robert": "dumitrescustefan/bert-base-romanian-cased-v1",
    "mbert":  "bert-base-multilingual-cased",
}[MODEL_KEY]

# SEED = 315
SEED = 222

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

BASE     = f"/content/drive/MyDrive/rodi_study/{RUN_NAME}"
CKPT_DIR = f"{BASE}/checkpoints"
RES_DIR  = f"{BASE}/results"
PRED_DIR = f"{BASE}/predictions"

for d in [CKPT_DIR, RES_DIR, PRED_DIR]:
    os.makedirs(d, exist_ok=True)

Mounted at /content/drive


In [ ]:
nodiac_train_file='/content/datasets/nodiac/train.json'
nodiac_validation_file='/content/datasets/nodiac/valid.json'
nodiac_test_file='/content/datasets/nodiac/test.json'

In [ ]:
def eval_ner(model, iterations, batch_size, accumulate_grad_batches):
  print("\033[92mPreparing environment ...\033[0m")
  !git clone https://github.com/Alexandra06T/RoDi.git
  !pip3 install -r RoDi/performance_analysis/evaluate/requirements.txt -q

  print("\033[92mRunning task ...\033[0m")
  !cd RoDi/performance_analysis/evaluate && python evaluate.py --batch_size=$batch_size --accumulate_grad_batches=$accumulate_grad_batches --model_name $model --train_file $nodiac_train_file --validation_file $nodiac_validation_file --test_file $nodiac_test_file --dirpath $CKPT_DIR --seed $SEED
  # !python evaluate.py --batch_size=$batch_size --accumulate_grad_batches=$accumulate_grad_batches --model_name $model --train_file $nodiac_train_file --validation_file $nodiac_validation_file --test_file $nodiac_test_file --dirpath $CKPT_DIR --seed $SEED

  return None


# configs
batch_size, accumulate_grad_batches = 8, 1
if "-large" in model or "-medium" in model:
  batch_size = 1
  accumulate_grad_batches = 8

In [ ]:
# run
if task == "Named Entity Recognition":
  eval_ner(model, iterations, batch_size, accumulate_grad_batches)

Streaming output truncated to the last 5000 lines.
                                                               0.151            
                                                               valid/strict:    
Epoch 11/999 ━╸━━━━━━━━━━━━━ 141/1125 0:00:40 •       3.51it/s v_num: 3.000     
                                      0:04:41                  valid/avg_loss:  
                                                               0.151            
                                                               valid/strict:    
Epoch 11/999 ━╸━━━━━━━━━━━━━ 142/1125 0:00:41 •       3.48it/s v_num: 3.000     
                                      0:04:43                  valid/avg_loss:  
                                                               0.151            
                                                               valid/strict:    
Epoch 11/999 ━╸━━━━━━━━━━━━━ 143/1125 0:00:41 •       3.48it/s v_num: 3.000     
                                      0:04:43             